# Базовые карты и графики

В этом ноутбуке мы познакомимся научимся создавать карты и графики с помощью базовых библиотек, а также размещать карту и диаграмму на одном изображении и синхронизировать цвета между разными визуализациями.

**Данные:**

- **spb_admin.gpkg** — полигональные данные о границах районов и округов Санкт-Петербурга.  
  Источник: материалы курса «Методы пространственного анализа», НИУ ВШЭ (Р. Гончаров)

- **spb_theaters.csv** — данные о театрах Санкт-Петербурга.  
  Источник: [Портал открытых данных Санкт-Петербурга](https://data.gov.spb.ru/)


## 0. Импорт библиотек


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily as ctx


## 1. Базовая карта: метод `.plot()`

В GeoPandas у каждого `GeoDataFrame` есть метод `.plot()`, с помощью которого можно создавать простые карты.


### 1.1. Загрузка данных


In [ ]:
districts = gpd.read_file("../data/spb_admin.gpkg", layer="district")
districts[["NAME", "Popul", "geometry"]].head()


### 1.2. Минимальная карта

Самый простой вызов `.plot()` работает без аргументов — GeoPandas выберет цвет и размер сам.


In [ ]:
districts.plot()
plt.show()

### 1.3. Настройка внешнего вида

#### Figure и Axes

В matplotlib любой рисунок состоит из двух объектов:

- **Figure** (`fig`) — «холст»: вся область рисунка целиком, включая поля и отступы.
- **Axes** (`ax`) — «кадр»: конкретная область, в которой строится карта или график. На одном холсте может быть несколько кадров.

`plt.subplots()` создаёт оба объекта сразу и возвращает их как пару `fig, ax`. После этого мы передаём `ax` в `.plot(ax=ax)` — так GeoPandas знает, _в какой именно кадр_ рисовать. Все дальнейшие настройки (заголовок, оси, размер) тоже делаются через `ax` или `fig`.

#### Основные параметры `.plot()`

| Параметр    | Что задаёт                    | Пример      |
| :---------- | :---------------------------- | :---------- |
| `color`     | Цвет заливки                  | `"#d0e8f1"` |
| `edgecolor` | Цвет границ                   | `"gray"`    |
| `linewidth` | Толщина границ                | `0.8`       |
| `alpha`     | Прозрачность (0–1)            | `0.7`       |
| `figsize`   | Размер холста в дюймах (ш, в) | `(8, 6)`    |
| `ax`        | Кадр, в который рисовать      | `ax`        |


In [ ]:
# Создаём холст (fig) размером 8×7 дюймов и один кадр (ax) внутри него
fig, ax = plt.subplots(figsize=(8, 7))

# Рисуем районы в кадр ax
districts.plot(
    ax=ax,
    color="#d0e8f1",      # цвет заливки
    edgecolor="#4a90a4",  # цвет границ
    linewidth=0.8,        # толщина границ
    alpha=0.9             # небольшая прозрачность
)

ax.set_title("Районы Санкт-Петербурга", fontsize=14, pad=12)  # заголовок карты
ax.axis("off")   # скрываем оси координат — на карте они не нужны

plt.tight_layout()  # убирает лишние поля вокруг рисунка
plt.show()          # отображает рисунок


### 1.4. Сохранение карты

Чтобы сохранить карту в файл, используем метод `fig.savefig()`.

По умолчанию файл сохраняется в ту же папку, где лежит ноутбук. Поддерживаемые форматы: `.png`, `.pdf`, `.svg` — формат определяется по расширению в имени файла.

Основные параметры:

| Параметр              | Что задаёт                          | Пример                                  |
| :-------------------- | :---------------------------------- | :-------------------------------------- |
| `dpi`                 | Разрешение в точках на дюйм         | `150` — экран, `300` — печать           |
| `bbox_inches="tight"` | Обрезает пустые поля вокруг рисунка | без него могут остаться широкие отступы |


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
districts.plot(ax=ax, color="#d0e8f1", edgecolor="#4a90a4", linewidth=0.8)
ax.set_title("Районы Санкт-Петербурга", fontsize=14)
ax.axis("off")
plt.tight_layout()

# Сохраняем холст в файл рядом с ноутбуком
# savefig нужно вызвать ДО plt.show() — после show() фигура очищается
fig.savefig("map_districts.png", dpi=150, bbox_inches="tight")

plt.show()


## 2. Точечные данные


### 2.1. Загрузим и подготовим точечные данные


In [ ]:
theaters_df = pd.read_csv("../data/spb_theaters.csv")

theaters = gpd.GeoDataFrame(
    theaters_df,
    geometry=gpd.points_from_xy(theaters_df["longitude"], theaters_df["latitude"]),
    crs="EPSG:4326"
)

theaters[["name", "type_simple", "geometry"]].head()


### 2.2. Карта точек

Чтобы разместить несколько слоёв на одной карте, нужно передать один и тот же `ax` в каждый вызов `.plot()`. Именно через `ax=ax` GeoPandas понимает, что нужно рисовать в уже существующий кадр, а не создавать новый.

```python
fig, ax = plt.subplots()

первый_слой.plot(ax=ax, ...)   # рисуем первым — он окажется снизу
второй_слой.plot(ax=ax, ...)   # рисуем вторым — он окажется сверху
```

Порядок имеет значение: каждый следующий слой перекрывает предыдущий.

Дополнительные параметры для точек:

| Параметр     | Что задаёт     | Примеры значений                                           |
| :----------- | :------------- | :--------------------------------------------------------- |
| `markersize` | Размер маркера | `5`, `20`, `50`                                            |
| `marker`     | Форма маркера  | `"o"` круг, `"s"` квадрат, `"^"` треугольник, `"*"` звезда |


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

# Слой 1: районы — фон, рисуем первыми
districts.plot(ax=ax, color="#f0f0f0", edgecolor="gray", linewidth=0.5)

# Слой 2: театры — точки поверх районов, в тот же кадр ax
theaters.plot(ax=ax, color="#d73027", markersize=20, marker="o", alpha=0.7)

ax.set_title("Театры Санкт-Петербурга", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()


### 2.3. Категориальная окраска

Чтобы окрасить объекты по значению столбца, передаём его в параметр `column=`. Для категориальных данных добавляем `categorical=True`.

| Параметр           | Что задаёт                        |
| :----------------- | :-------------------------------- |
| `column`           | Поле с данными.                   |
| `categorical=True` | Трактовать значения как категории |
| `legend=True`      | Показать легенду                  |
| `cmap`             | Цветовая схема                    |
| `legend_kwds`      | Параметры легенды (словарь)       |

#### Цветовые схемы (colormap)

Все цветовые схемы matplotlib делятся на три типа:

| Тип                  | Когда использовать                                   | Примеры                                   |
| :------------------- | :--------------------------------------------------- | :---------------------------------------- |
| **Качественные**     | Категории без порядка (типы, классы)                 | `Set1`, `Set2`, `Set3`, `tab10`, `Paired` |
| **Последовательные** | Числовые данные от меньшего к большему               | `Blues`, `BuPu`, `YlOrRd`, `viridis`      |
| **Расходящиеся**     | Данные с нейтральной серединой (отклонение от нормы) | `RdBu`, `coolwarm`, `PiYG`                |

Для категориальных карт подходят **качественные** схемы — в них цвета максимально отличаются друг от друга.

Полный список доступных палитр можно посмотреть на [сайте matplotlib](https://matplotlib.org/stable/gallery/color/colormap_reference.html) или вывести прямо в Python:


In [ ]:
print(plt.colormaps())  # список всех доступных имён

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

districts.plot(ax=ax, color="#f5f5f5", edgecolor="gray", linewidth=0.4)
theaters.plot(
    ax=ax,
    column="type_simple",   # поле, по которому окрашиваем
    categorical=True,       # значения — категории, не числа
    legend=True,            # автоматически создаёт легенду по уникальным значениям столбца
    cmap="Set2",
    markersize=35,
    alpha=0.85,
    legend_kwds={           # словарь с настройками легенды
        "loc": "lower left",        # положение на карте
        "title": "Тип учреждения",  # заголовок легенды
        "framealpha": 0.8,          # прозрачность фона легенды (0 — прозрачный, 1 — глухой)
    }
)

ax.set_title("Театры Санкт-Петербурга по типу учреждения", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Статистические графики

Карта хорошо показывает, _где_ что-то происходит, но плохо справляется со сравнением и долями. Для этого используют статистические графики — они дополняют карту и вместе дают полную картину.

В этом разделе построим два типа диаграмм с помощью `matplotlib`.


### 3.1. Столбчатая диаграмма

**Столбчатая диаграмма** используется для сравнения значений по категориям.

В matplotlib два варианта:

- `ax.bar(x, height)` — вертикальные столбцы
- `ax.barh(y, width)` — горизонтальные столбцы

Горизонтальные удобнее, когда подписи категорий длинные — они не налезают друг на друга.

Перед построением нужно посчитать количество объектов в каждой категории — для этого используем `value_counts()`.


In [ ]:
# value_counts() считает количество вхождений каждого значения
# и возвращает Series: индекс — категория, значение — количество
type_counts = theaters["type_simple"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))

# barh(y, width): y — подписи по вертикали, width — длина столбца
ax.barh(type_counts.index, type_counts.values, color="#4a90a4")

ax.set_xlabel("Количество театров")
ax.set_title("Театры по типу учреждения")

# Убираем верхнюю и правую рамку — они не несут информации
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()


### 3.2. Круговая диаграмма

**Круговая диаграмма** показывает доли категорий от целого. Хорошо работает, когда категорий немного (3–5) и важно подчеркнуть соотношение частей.

Основные параметры `ax.pie()`:

| Параметр     | Что задаёт                                                        |
| :----------- | :---------------------------------------------------------------- |
| `autopct`    | Подпись с процентом на каждом секторе (`"%1.0f%%"` — целое число) |
| `startangle` | Угол начала первого сектора (90 — от верхней точки)               |
| `colors`     | Список цветов для секторов                                        |
| `wedgeprops` | Словарь с оформлением секторов (граница, толщина и т.д.)          |


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.pie(
    type_counts.values,
    labels=type_counts.index,
    colors=["#4a90d9", "#e8714a", "#5cb98c"],  # цвета задаём вручную
    autopct="%1.0f%%",       # подписи с процентами (0 знаков после запятой)
    startangle=90,           # начинаем с верхней точки
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}  # белые границы между секторами
)

ax.set_title("Структура театров по типу учреждения")
plt.tight_layout()
plt.show()


## 4. Компоновка: карта и график рядом

До этого мы создавали по одному кадру на холсте. `plt.subplots(nrows, ncols)` позволяет создать сетку кадров — и разместить карту и диаграмму рядом на одном изображении.

```
fig
┌─────────────────────────────────┐
│  ax_map          │  ax_chart    │
│  (карта)         │  (диаграмма) │
└─────────────────────────────────┘
```

`plt.subplots(1, 2)` означает: **1 строка, 2 столбца**. В результате получаем:

- `fig` — один общий холст
- два кадра, которые можно распаковать в отдельные переменные: `fig, (ax_map, ax_chart) = ...`

Каждый кадр дальше используется независимо: в `ax_map` рисуем карту, в `ax_chart` — диаграмму.


### 4.1. Простая компоновка

Обратите внимание: в этом примере карта и диаграмма строятся независимо и используют разные цвета для одних и тех же категорий. В следующем разделе мы это исправим.


In [ ]:
# 1 строка, 2 столбца — два кадра на одном холсте
# figsize=(14, 6) — общий размер всего холста (оба кадра вместе)
fig, (ax_map, ax_chart) = plt.subplots(1, 2, figsize=(14, 6))

# Левый кадр: карта
districts.plot(ax=ax_map, color="#f0f0f0", edgecolor="gray", linewidth=0.4)
theaters.plot(ax=ax_map, column="type_simple", categorical=True,
              cmap="Set2", markersize=30, alpha=0.85)
ax_map.set_title("Театры по типу учреждения", fontsize=12)
ax_map.axis("off")

# Правый кадр: диаграмма
type_counts = theaters["type_simple"].value_counts()
ax_chart.barh(type_counts.index, type_counts.values)
ax_chart.set_title("Количество театров по типу", fontsize=12)
ax_chart.set_xlabel("Количество")
ax_chart.spines[["top", "right"]].set_visible(False)  # убирает верхнюю и правую рамку

plt.tight_layout()
plt.show()


### 4.2. Синхронизация цветов

В примере 4.1 карта и диаграмма назначают цвета независимо друг от друга — каждая в своём порядке. В итоге «Городское гос.» на карте может быть зелёным, а на диаграмме — оранжевым. Легенда и диаграмма будут противоречить друг другу.

**Решение — словарь `color_dict`.**

Мы сами один раз решаем, какой цвет соответствует каждой категории, и записываем это в словарь. Затем и карта, и диаграмма берут цвета из этого словаря — и всегда получают одинаковый результат.

```
color_dict = {
    "Городское гос.":   <цвет 1>,
    "Федеральное":      <цвет 2>,
    "Негосударственное": <цвет 3>,
}
```

Категория → цвет. Всегда одно и то же соответствие.


In [ ]:
color_dict = {
    "Городское гос.":    "#4a90d9",  # голубой
    "Федеральное":       "#e8714a",  # терракотовый
    "Негосударственное": "#5cb98c",  # мятный
}


Словарь готов. Теперь нужно применить его в двух местах.

**На карте** — передаём цвет прямо в параметр `color=` с помощью метода `.map()`:

```python
theaters.plot(ax=ax_map, color=theaters["type_simple"].map(color_dict), ...)
```

**На диаграмме** — создаём список цветов в том же порядке, в каком идут категории:

```python
bar_colors = [color_dict[cat] for cat in type_counts.index]
ax_chart.barh(..., color=bar_colors)
```

Теперь «Городское гос.» получит один и тот же цвет везде.


#### Метод `.map()`

`.map()` — метод pandas, который проходит по каждому значению в столбце и заменяет его на что-то другое по заданному правилу. Правилом может быть словарь.

```
theaters["type_simple"]        →   theaters["type_simple"].map(color_dict)

"Городское гос."               →   "#4a90d9"
"Федеральное"                  →   "#e8714a"
"Городское гос."               →   "#4a90d9"
"Негосударственное"            →   "#5cb98c"
...                                ...
```

Результат — столбец с цветами, по одному на каждый театр. Именно его мы передаём в `color=`.


In [ ]:
type_counts = theaters["type_simple"].value_counts()

fig, (ax_map, ax_chart) = plt.subplots(1, 2, figsize=(14, 6))

# Карта
districts.plot(ax=ax_map, color="#f5f5f5", edgecolor="gray", linewidth=0.4)
theaters.plot(
    ax=ax_map,
    color=theaters["type_simple"].map(color_dict),
    markersize=35,
    alpha=0.9,
    edgecolor="white",  # тонкая белая обводка вокруг каждой точки
    linewidth=0.4
)
ax_map.set_title("Театры по типу учреждения", fontsize=12)
ax_map.axis("off")

# Диаграмма
bar_colors = [color_dict[cat] for cat in type_counts.index]
ax_chart.barh(type_counts.index, type_counts.values, color=bar_colors)
ax_chart.set_title("Количество театров по типу", fontsize=12)
ax_chart.set_xlabel("Количество")
ax_chart.spines[["top", "right"]].set_visible(False)  # убирает верхнюю и правую рамку

plt.tight_layout()
plt.show()


## 5. Картографическая подложка

На некоторых картах может пригодиться контекст, чтобы понимать, где находится территория. По аналогии с веб-картами и на статичные карты добавлять растровые подложки Для этого используется библиотека `contextily`.

Из особенностей Тайловые сервисы работают в проекции **Web Mercator (EPSG:3857)**, поэтому данные нужно предварительно перепроецировать.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

# Перепроецируем данные в Web Mercator
districts_3857 = districts.to_crs(epsg=3857)
theaters_3857  = theaters.to_crs(epsg=3857)

# Рисуем данные
districts_3857.plot(ax=ax, color="none", edgecolor="#666666", linewidth=0.5, alpha=0.6)
theaters_3857.plot(
    ax=ax,
    color=theaters["type_simple"].map(color_dict),
    markersize=35,
    alpha=0.9,
    edgecolor="white",
    linewidth=0.4
)

# Добавляем подложку
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_title("Театры Санкт-Петербурга", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

# Другие популярные подложки:
# ctx.providers.OpenStreetMap.Mapnik     — цветная, детальная
# ctx.providers.CartoDB.Positron         — светлая, минималистичная (рекомендуется)
# ctx.providers.CartoDB.DarkMatter       — тёмная


## Итог

В этом ноутбуке мы научились:

- строить базовые карты с помощью `.plot()` и настраивать их внешний вид;
- размещать точечные данные на карте и окрашивать их по категории;
- строить столбчатые и круговые диаграммы с `matplotlib`;
- совмещать карту и диаграмму на одной фигуре и **синхронизировать цвета** через словарь `color_dict`;
- добавлять картографическую подложку с помощью `contextily`.
